# Expanded Exercise: Checking Assumptions for t-tests, ANOVA & Tukey
## Practical Workflow with Decision Framework, Simulation & Real Data Application

Before running any parametric test (two-sample t-test, ANOVA, or Tukey), we must verify key assumptions. This exercise turns the vague checklist into a **practical, decision-oriented workflow** you can use on real projects.

You will work with two new distributions (`dist_1` from `1.csv` and `dist_2` from `2.csv`) and also apply the checks to the VeryAnts data from previous exercises.


## Flowchart: Assumption Checking Workflow (Practical Decision Tree)
```mermaid
flowchart TD
    Start[Load Data & Define Groups] --> VarCheck{Check Equal Variances<br/>Ratio of SDs ~0.9-1.1?<br/>+ Levene test p > 0.05?}
    VarCheck -->|Yes| NormCheck{Check Normality<br/>Shapiro p > 0.05 or large n + visual OK?}
    VarCheck -->|No| Welch[Use Welch t-test<br/>(equal_var=False) or<br/>non-parametric test]
    NormCheck -->|Yes| Proceed[Proceed with t-test / ANOVA / Tukey]
    NormCheck -->|No| Robust[Large n? → CLT often OK<br/>Small n? → Transform data or use non-parametric]
    Robust --> Proceed
    Proceed --> Report[Document checks in report<br/>+ sensitivity analysis if assumptions borderline]
    Report --> Audience[Tailor depth to audience<br/>Technical: full tests + plots<br/>Execs: 'Assumptions checked, results reliable']
```
**Practical rule of thumb:** With n ≥ 30–50 per group, moderate violations are often tolerable due to the Central Limit Theorem. Always report what you checked.


## 1. Load Data and Quick Exploration

**Instructions:**
1. Load `dist_1` from `1.csv` and `dist_2` from `2.csv` using `np.genfromtxt`
2. Print shape, mean, and standard deviation for each
3. Calculate the ratio of standard deviations (`dist_1.std() / dist_2.std()`)
4. Create overlaid histograms (as in the original prompt)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

# TODO: Load the two distributions
dist_1 = np.genfromtxt('1.csv')
dist_2 = np.genfromtxt('2.csv')

print('dist_1: n =', len(dist_1), 'mean =', round(dist_1.mean(), 2), 'std =', round(dist_1.std(), 2))
print('dist_2: n =', len(dist_2), 'mean =', round(dist_2.mean(), 2), 'std =', round(dist_2.std(), 2))

# TODO: Ratio of standard deviations
ratio = dist_1.std() / dist_2.std()
print('Ratio of std (dist_1 / dist_2) =', round(ratio, 3))

# TODO: Overlaid histogram
plt.figure(figsize=(8,5))
plt.hist(dist_1, alpha=0.6, bins=20, label='dist_1', density=True)
plt.hist(dist_2, alpha=0.6, bins=20, label='dist_2', density=True)
plt.legend()
plt.title('Distribution of dist_1 and dist_2')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()


## 2. Check Equal Variances (More Rigorously)

**Instructions:**
1. Calculate the ratio of standard deviations (already done above)
2. Run Levene’s test: `stats.levene(dist_1, dist_2)`
3. In a markdown cell: Is the equal variance assumption reasonably met? What should you do if it is violated?


In [ ]:
# TODO: Levene's test for equal variances
levene_result = stats.levene(dist_1, dist_2)
print('Levene test statistic:', round(levene_result.statistic, 3))
print('Levene p-value:', round(levene_result.pvalue, 4))

# TODO: Decision in markdown cell below


## 3. Check Normality

**Instructions:**
1. Create Q-Q plots for both distributions
2. Run Shapiro-Wilk test on each: `stats.shapiro()`
3. Visually inspect the overlaid histogram you created earlier
4. Decide: Are the distributions approximately normal? Consider sample size (n=100 each).


In [ ]:
# TODO: Q-Q plots
fig, axes = plt.subplots(1, 2, figsize=(10,4))
stats.probplot(dist_1, dist='norm', plot=axes[0])
axes[0].set_title('Q-Q Plot: dist_1')
stats.probplot(dist_2, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: dist_2')
plt.tight_layout()
plt.show()

# TODO: Shapiro-Wilk tests
print('Shapiro-Wilk dist_1 p-value:', round(stats.shapiro(dist_1).pvalue, 4))
print('Shapiro-Wilk dist_2 p-value:', round(stats.shapiro(dist_2).pvalue, 4))


## 4. Decision Framework & What to Do When Assumptions Are Violated

**Instructions:**
Based on your checks above, answer in a markdown cell:
1. Can we safely run a standard two-sample t-test (equal variances) on these data?
2. If not, what are our options? (Welch t-test, non-parametric test, transform data, proceed with caution due to large n)
3. How would you report these checks in a data analysis report?


## 5. Apply the Same Checks to Previous Data (VeryAnts)

**Instructions (More Practice):**
Load the `veryants.csv` file from earlier exercises.
Check the equal variance and normality assumptions for the three stores (A, B, C).
Would you proceed with ANOVA + Tukey, or make adjustments?


In [ ]:
# TODO: Load veryants and check assumptions for the three stores
# (You can reuse code from previous notebooks)
print('Apply assumption checks to the VeryAnts data from previous exercises.')


## 6. Simulation: Impact of Violated Assumptions

**Goal:** See what happens to a t-test when variances are very different or data is skewed.

Modify the parameters and observe how the p-value and Type I error rate change.


In [ ]:
np.random.seed(42)

# === MODIFIABLE PARAMETERS ===
n = 100
mean1, mean2 = 18, 12
std1, std2 = 3, 5          # try making std2 much larger (e.g. 8) to violate equal variance
n_simulations = 500
alpha = 0.05

# === Simulation skeleton ===
significant_count = 0

for i in range(n_simulations):
    g1 = np.random.normal(mean1, std1, n)
    g2 = np.random.normal(mean2, std2, n)
    _, p = stats.ttest_ind(g1, g2, equal_var=True)   # try equal_var=False too
    if p < alpha:
        significant_count += 1

print('Proportion of simulations with p < 0.05 (when means differ):', round(significant_count / n_simulations, 3))
print('Try changing std2 to a much larger value and re-run to see the effect of violated equal variance.')


## 7. Conclusion & Audience-Aware Reporting of Assumption Checks

Write a short section suitable for a data analysis report that documents the assumption checks you performed.
Create versions for different audiences:
- Technical supervisor: full tests, p-values, plots, decision
- Executive: high-level statement (“Assumptions were verified; results are reliable” or “We used a more robust Welch t-test because variances differed”)
